## This notebook can be used to rank a list of nodes from a category that connect to an entity such as a gene. 

In [1]:

import sys
import os
sys.path.append('../TCT/')
from TCT import node_normalizer
from TCT import name_resolver
from TCT import translator_metakg
from TCT import translator_kpinfo
from TCT import translator_query
from TCT import TCT



### Load Translator resources


In [2]:
APInames, metaKG, Translator_KP_info = translator_metakg.load_translator_resources()

Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}
Skipping server without x-maturity: {'url': '/sipr'}


In [ ]:
#Translator_KP_info

In [3]:
All_predicates = list(set(metaKG['Predicate']))
All_categories = list((set(list(set(metaKG['Subject']))+list(set(metaKG['Object'])))))
API_withMetaKG = list(set(metaKG['API']))

# generate a dictionary of API and its predicates
API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(metaKG[metaKG['API'] == api]['Predicate']))

## Find the neighborhood of an entity from a subset of APIs 


In [4]:
# select a list of APIs to use and a list of predicates to use, if the list is empty, use all APIs and predicates
#selected_APIlist = ['Microbiome KP - TRAPI 1.5.0']
selected_APIlist = []
if len(selected_APIlist) == 0:
    select_APIs = APInames
else:
    select_APIs = {k: APInames[k] for k in selected_APIlist if k in APInames}

selected_metaKG = metaKG[metaKG['API'].isin(select_APIs.keys())]
print(select_APIs)
print(selected_metaKG.shape)


{'BioThings Explorer (BTE) TRAPI': 'https://bte.transltr.io/v1/query/', 'COHD TRAPI': 'https://cohd-api.transltr.io/api/query/', 'Drug Approvals KP - TRAPI 1.5.0': 'https://multiomics.rtx.ai:9990/dakp/query', 'CATRAX BigGIM DrugResponse Performance Phase KP - TRAPI 1.5.0': 'https://multiomics.rtx.ai:9990/BigGIM_DrugResponse_PerformancePhase/query', 'ARAX Translator Reasoner - TRAPI 1.6.0': 'https://arax.transltr.io/api/arax/v1.4/query/', 'Automat-binding-db(Trapi v1.5.0)': 'https://automat.renci.org/binding-db/query/', 'Cqs(Trapi v1.5.0)': 'https://cqs-dev.apps.renci.org/query/', 'Text Mined Cooccurrence API': 'https://cooccurrence.ci.transltr.io/query/', 'Genetics Data Provider for NCATS Biomedical Translator Reasoners': 'https://genetics-kp.transltr.io/genetics_provider/trapi/v1.5/query/', 'Knowledge Collaboratory API': 'https://collaboratory-api.transltr.io/query/', 'CATRAX Pharmacogenomics KP - TRAPI 1.5.0': 'https://multiomics.rtx.ai:9990/PharmacogenomicsKG/query', 'Clinical Trial

In [5]:
name_resolver.lookup('acute myeloid leukemia', return_top_response=False, biolink_type='biolink:Disease',  limit=100) # sometimes the identifiers are not in the top 1, users need to check the other returned results


[TranslatorNode(curie='MONDO:0018874', label='acute myeloid leukemia', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]),
 TranslatorNode(curie='MONDO:0005223', label='acute myeloid leukemia with minimal differentiation', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]),
 TranslatorNode(curie='MONDO:0017893', label='inherited acute myeloid leukemia', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]),
 TranslatorNode(curie='MONDO:0020317', label='acute myeloid leukemia with 11q23 ab

In order to use the neighborhood finder, we have to look up a CURIE ID for a given term.

In [7]:
#name_resolver.lookup('BCL2')
#name_resolver.lookup('BCL2', return_top_response=False, biolink_type='biolink:Gene',  limit=100, only_taxa='NCBITaxon:9606') # sometimes the identifiers are not in the top 1, users need to check the other returned results

input_identifiers = 'MONDO:0018874'


Neighborhood finder identifies all nodes *b* that are connected to the given node *a*, where *b* is part of a defined list of categories - returning the neighborhood of node *a*.

In [6]:
input_node_id, result, result_parsed, result_ranked_by_primary_infores = TCT.Neighborhood_finder('MONDO:0018874',
                                                                                            node2_categories = ['biolink:Disease'],
                                                                                            APInames = select_APIs,
                                                                                            metaKG = selected_metaKG,
                                                                                            API_predicates = API_predicates)     

MONDO:0018874
Automat-robokop(Trapi v1.5.0): Success!
RTX KG2 - TRAPI 1.5.0: Success!
Automat-monarchinitiative(Trapi v1.5.0): Success!
Automat-drug-central(Trapi v1.5.0): Success!
Automat-hetionet(Trapi v1.5.0): Success!
Automat-pharos(Trapi v1.5.0): Success!
Automat-ctd(Trapi v1.5.0): Success!
Automat-reactome(Trapi v1.5.0): Success!
Automat-ubergraph(Trapi v1.5.0): Success!
Automat-gwas-catalog(Trapi v1.5.0): Success!
Service Provider TRAPI: Success!
BioThings Explorer (BTE) TRAPI: Success!
orphanet:98277: no preferred name
orphanet:524: no preferred name
orphanet:98827: no preferred name
NodeNorm does not know about these identifiers: MONDO:0975868,MONDO:0975870,MONDO:0975871,MONDO:0975872,OMIM:MTHU010293,OMIM:MTHU020408,OMIM:MTHU020414,OMIM:MTHU034946,CHV:0000007343,UMLS:C2826176,CHV:0000050375,CHV:0000007348,DOID:0070633,DOID:0070632,DOID:0070631,DOID:0070630,DOID:0070629,HP:0000001,HP:0011016,MONDO:0009410,MONDO:0004721,MONDO:0007140,UMLS:C0524869,MONDO:0020315


In [8]:
# write a result to a json file
import json
with open('TCT_neighborhood_finder_result_'+input_identifiers.replace(':', '_')+'.json', 'w') as f:
    json.dump(result, f)

In [ ]:
# Step 8: Visualize the results
TCT.visulization_one_hop_ranking(result_ranked_by_primary_infores, result_parsed, 
                                num_of_nodes = 50, input_query = input_node_id, 
                                fontsize = 5)

In [ ]:
result_ranked_by_primary_infores

In [ ]:
from TCT import TCT_Visualization

dic_graph = TCT_Visualization.visualize_neighborhood_graph(result, show_label=True, height="500", width="100%")

In [ ]:
# End of the example
